In [1]:
import scipy as sp
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
from pathlib import Path  
import scanpy as sc
import os 
from model_evaluation import *

sc.settings.set_figure_params(figsize=(3, 3))

In [2]:
import numpy as np
import pandas as pd

import scanpy as sc

import scipy as sp
from sklearn.metrics import precision_recall_curve, auc

def compute_precision_and_recall_auc(predicted, true):
    # Compute precision and recall curve and AUC
    precision, recall, _ = precision_recall_curve(true, predicted)
    pr_auc = auc(recall, precision)
    return pr_auc

def cluster_metrics(abs_ratios, ratios, is_abundant, leiden_labels):
    # Non-abundant clusters should always be lower 
    non_abundant_over_abundant = abs_ratios[is_abundant].mean() / abs_ratios[~is_abundant].mean()
    # Compute fraction correct sign
    ratios_abundant = ratios[is_abundant]  # Only clusters 1 and 2 
    leiden_abundant = np.array(leiden_labels)[is_abundant]

    leiden_abundant_unique = np.unique(leiden_abundant)
    assert "0" not in leiden_abundant_unique
    assert "3" not in leiden_abundant_unique
    
    ratios_abundant[leiden_abundant=="2"] = ratios_abundant[leiden_abundant=="2"] * (-1)
    correct_sign_prop = (ratios_abundant >= 0).sum() / len(ratios_abundant)
    
    # Kolmogorov-Smirnov test 
    ks_D, _ = sp.stats.ks_2samp(
        abs_ratios[is_abundant],
        abs_ratios[~is_abundant],
        alternative="two-sided",
        mode="asymp"
    )
    w_distance = sp.stats.wasserstein_distance(abs_ratios[is_abundant], abs_ratios[~is_abundant])
    return non_abundant_over_abundant, correct_sign_prop, ks_D, w_distance
    
def compute_evaluation_metrics(result_path, model, subsampling_rates, n_runs):
    results_per_oversamp = {
        "p_major_cluster": [], 
        "p_minor_cluster": [],
        "auc_score": [], 
        "non_abundant_over_abundant": [],
        "correct_sign_prop": [],
        "ks_D": [], 
        "w_distance": [],
        "run": []
        }
    
    results_per_run = {
        "metric": [], 
        "value": [],
        "run": []
        }

    for i in range(n_runs):
        # Compute the ratios and abundances per cluster
        per_cluster_ratios = []
        per_cluster_log_odds = []
        
        for r_oversamp in subsampling_rates:
            # Compute the true per-cluster ratio 
            p = 0.5 - r_oversamp  # Always minor cluster prop
            one_minus_p = 1 - p  # Always major cluster prop
    
            # Collect results and compute AUC 
            if model=="mrvi":
                adata_generated = sc.read_h5ad(result_path / f"oversamp_{r_oversamp}_{i}" / f"oversamp_{r_oversamp}_{i}.h5ad")
            elif model=="meld":
                adata_generated = sc.read_h5ad(result_path / f"oversamp_{r_oversamp}.h5ad")
            else:
                adata_generated = sc.read_h5ad(result_path / f"oversamp_{r_oversamp}_{i}.h5ad")
            
            # Collect abundance labels 
            is_abundant = np.logical_or(adata_generated.obs.leiden=="1", adata_generated.obs.leiden=="2")
            if model=="mrvi":
                abs_ratio = np.abs(adata_generated.obs[f"log_ratio_oversamp_{r_oversamp}_{i}"]).values
                ratio = adata_generated.obs[f"log_ratio_oversamp_{r_oversamp}_{i}"].values
            elif model=="scRatio":
                abs_ratio = np.abs(adata_generated.obs["log_ratios"]).values
                ratio = adata_generated.obs["log_ratios"].values
            else:
                abs_ratio = np.abs(adata_generated.obs["log_ratio"]).values
                ratio = adata_generated.obs["log_ratio"].values

            if model=="milo":
                adata_generated = adata_generated[~np.isnan(abs_ratio)]
                is_abundant = is_abundant[~np.isnan(abs_ratio)]
                abs_ratio = abs_ratio[~np.isnan(abs_ratio)]
                ratio = ratio[~np.isnan(ratio)]

            # Compute AUC
            auc = compute_precision_and_recall_auc(abs_ratio, is_abundant)
            results_per_oversamp["p_major_cluster"].append(one_minus_p)
            results_per_oversamp["p_minor_cluster"].append(p)
            results_per_oversamp["auc_score"].append(auc)
            results_per_oversamp["run"].append(i)
            
            for cl in ["0", "1", "2", "3"]:
                per_cluster_ratios.append(np.mean(ratio[adata_generated.obs["leiden"]==cl]))

                numerator = (adata_generated.obs.loc[adata_generated.obs["leiden"]==cl, "treatment"]==1).sum()
                denominator = (adata_generated.obs.loc[adata_generated.obs["leiden"]==cl, "treatment"]==0).sum() 

                abs_log_odds = np.log((numerator + 1e-9) / (denominator + 1e-9))
                per_cluster_log_odds.append(abs_log_odds)
            
            # Compute cluster metrics
            abundant_over_non_abundant, correct_sign_prop, ks_D, w_distance = cluster_metrics(abs_ratio, ratio, is_abundant, adata_generated.obs["leiden"])
            results_per_oversamp["non_abundant_over_abundant"].append(abundant_over_non_abundant)
            results_per_oversamp["correct_sign_prop"].append(correct_sign_prop)
            results_per_oversamp["ks_D"].append(ks_D)
            results_per_oversamp["w_distance"].append(w_distance)
        
        results_per_run["metric"].append("spearman_log_odds")
        results_per_run["run"].append(i)
        results_per_run["value"].append(sp.stats.spearmanr(per_cluster_ratios, per_cluster_log_odds)[0])
        results_per_run["metric"].append("pearson_log_odds")
        results_per_run["run"].append(i)
        results_per_run["value"].append(sp.stats.pearsonr(per_cluster_ratios, per_cluster_log_odds)[0])

    # After computing the metrics per run and abundance, compute metrics per run only. 
    results_per_oversamp_df = pd.DataFrame(results_per_oversamp)
    for i in range(n_runs):
        # Take df at run i 
        results_per_oversamp_df_i = results_per_oversamp_df.loc[results_per_oversamp_df.run==i]
        for metric in results_per_oversamp_df_i.columns:
            name = f"corr_{metric}_p"
            results_per_run["metric"].append(name)
            results_per_run["run"].append(i) 
            results_per_run["value"].append(sp.stats.spearmanr(results_per_oversamp_df_i[metric],
                                                 results_per_oversamp_df_i["p_major_cluster"])[0])

            name = f"mean_{metric}"
            results_per_oversamp_df_over_i_03 = results_per_oversamp_df_i[results_per_oversamp_df_i.p_major_cluster>0.8]
            results_per_run["metric"].append(name)
            results_per_run["run"].append(i) 
            results_per_run["value"].append(results_per_oversamp_df_over_i_03[metric].mean())
        
    # Results per data frame 
    results_per_run_df = pd.DataFrame(results_per_run)    
    return results_per_oversamp_df, results_per_run_df


Initialize subsampling rates

In [3]:
subsampling_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.45, 0.50]
# subsampling_rates = [0.10, 0.30, 0.45]

## Evaluate Mrvi 

In [4]:
mrvi_result_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/mrvi")
results_mrvi_per_oversamp, results_mrvi_per_run = compute_evaluation_metrics(mrvi_result_path, "mrvi", subsampling_rates, 3)

/tmp/ipykernel_793092/2231908646.py:132: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  results_per_run["value"].append(sp.stats.spearmanr(results_per_oversamp_df_i[metric],


In [5]:
results_mrvi_per_run

,metric,value,run
0,spearman_log_odds,0.851906,0
1,pearson_log_odds,0.665817,0
2,spearman_log_odds,0.855938,1
3,pearson_log_odds,0.703736,1
4,spearman_log_odds,0.863270,2
5,pearson_log_odds,0.636894,2
6,corr_p_major_cluster_p,1.000000,0
7,mean_p_major_cluster,0.950000,0
8,corr_p_minor_cluster_p,-1.000000,0
9,mean_p_minor_cluster,0.050000,0


In [6]:
results_mrvi_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1)

,,auc_score,non_abundant_over_abundant,correct_sign_prop,ks_D,w_distance
p_major_cluster,p_minor_cluster,,,,,
0.50,0.50,0.465574,2.420448,0.408618,0.259781,0.082087
0.55,0.45,0.657164,3.474353,0.856563,0.510249,0.162138
0.60,0.40,0.805004,4.966484,0.949816,0.744516,0.262006
0.70,0.30,0.872288,5.773610,0.971775,0.836960,0.519815
0.80,0.20,0.901050,6.699221,0.982648,0.853962,0.833159
0.90,0.10,0.900619,6.335619,0.979171,0.848441,1.193009
0.95,0.05,0.891887,7.603555,0.974529,0.808235,1.453113
1.00,0.00,0.862672,8.047257,0.969957,0.793132,1.465103


In [7]:
results_mrvi_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1).mean(0)

auc_score                     0.794532
non_abundant_over_abundant    5.665069
correct_sign_prop             0.886635
ks_D                          0.706909
w_distance                    0.746304
dtype: float64

In [8]:
results_mrvi_per_run.groupby("metric").mean()

,value,run
metric,,
corr_auc_score_p,0.714286,1.0
corr_correct_sign_prop_p,0.674603,1.0
corr_ks_D_p,0.492063,1.0
corr_non_abundant_over_abundant_p,0.833333,1.0
corr_p_major_cluster_p,1.000000,1.0
corr_p_minor_cluster_p,-1.000000,1.0
corr_run_p,NaN,1.0
corr_w_distance_p,0.984127,1.0
mean_auc_score,0.885059,1.0


In [9]:
results_mrvi_per_run.groupby("metric").std()/np.sqrt(3)

,value,run
metric,,
corr_auc_score_p,0.090141,0.57735
corr_correct_sign_prop_p,0.055556,0.57735
corr_ks_D_p,0.093570,0.57735
corr_non_abundant_over_abundant_p,0.013746,0.57735
corr_p_major_cluster_p,0.000000,0.57735
corr_p_minor_cluster_p,0.000000,0.57735
corr_run_p,NaN,0.57735
corr_w_distance_p,0.007937,0.57735
mean_auc_score,0.003691,0.57735


## Evaluate Milo 

In [10]:
# milo_result_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/milo")
# results_milo_per_oversamp, results_milo_per_run = compute_evaluation_metrics(milo_result_path, "milo", subsampling_rates, 3)

In [11]:
# results_milo_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1)

In [ ]:
# results_milo_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1).mean(0)

In [ ]:
# results_milo_per_run.groupby("metric").mean()

In [ ]:
# results_milo_per_run.groupby("metric").std()/np.sqrt(3)

## Evaluate MELD

In [13]:
meld_result_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/meld")
results_meld_per_oversamp, results_meld_per_run = compute_evaluation_metrics(meld_result_path, "meld", subsampling_rates, 3)

/tmp/ipykernel_793092/2231908646.py:132: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  results_per_run["value"].append(sp.stats.spearmanr(results_per_oversamp_df_i[metric],


In [14]:
results_meld_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1)

,,auc_score,non_abundant_over_abundant,correct_sign_prop,ks_D,w_distance
p_major_cluster,p_minor_cluster,,,,,
0.50,0.50,0.629607,3.582369,0.021553,0.845292,0.006299
0.55,0.45,0.925110,6.923053,0.996399,0.819965,0.047419
0.60,0.40,0.935041,6.934956,0.996346,0.813418,0.100588
0.70,0.30,0.933073,6.125755,0.996081,0.805553,0.201706
0.80,0.20,0.932196,6.254434,0.996240,0.806972,0.314571
0.90,0.10,0.933963,6.383198,0.996240,0.807507,0.424336
0.95,0.05,0.930835,6.431556,0.996452,0.810150,0.489415
1.00,0.00,0.934540,6.540709,0.996240,0.807468,0.554932


In [15]:
results_meld_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1).mean(0)

auc_score                     0.894296
non_abundant_over_abundant    6.147004
correct_sign_prop             0.874444
ks_D                          0.814540
w_distance                    0.267408
dtype: float64

In [16]:
results_meld_per_run.groupby("metric").mean()

,value,run
metric,,
corr_auc_score_p,0.476190,1.0
corr_correct_sign_prop_p,0.268373,1.0
corr_ks_D_p,-0.595238,1.0
corr_non_abundant_over_abundant_p,0.166667,1.0
corr_p_major_cluster_p,1.000000,1.0
corr_p_minor_cluster_p,-1.000000,1.0
corr_run_p,NaN,1.0
corr_w_distance_p,1.000000,1.0
mean_auc_score,0.933113,1.0


In [17]:
results_meld_per_run.groupby("metric").std()/np.sqrt(3)

,value,run
metric,,
corr_auc_score_p,0.00000,0.57735
corr_correct_sign_prop_p,0.00000,0.57735
corr_ks_D_p,0.00000,0.57735
corr_non_abundant_over_abundant_p,0.00000,0.57735
corr_p_major_cluster_p,0.00000,0.57735
corr_p_minor_cluster_p,0.00000,0.57735
corr_run_p,NaN,0.57735
corr_w_distance_p,0.00000,0.57735
mean_auc_score,0.00000,0.57735


## Deterministic

In [18]:
scratio_det_result_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/scRatio/original_runs/deterministic")
results_scratio_det_per_oversamp, results_scratio_det_per_run = compute_evaluation_metrics(scratio_det_result_path, "scRatio", subsampling_rates, 3)

/tmp/ipykernel_793092/2231908646.py:132: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  results_per_run["value"].append(sp.stats.spearmanr(results_per_oversamp_df_i[metric],


In [19]:
results_scratio_det_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1)

,,auc_score,non_abundant_over_abundant,correct_sign_prop,ks_D,w_distance
p_major_cluster,p_minor_cluster,,,,,
0.50,0.50,0.309107,1.080800,0.444821,0.041957,0.005554
0.55,0.45,0.455529,1.563570,0.832098,0.251910,0.047998
0.60,0.40,0.658027,2.337031,0.926040,0.479445,0.167700
0.70,0.30,0.835349,3.130810,0.971404,0.693732,0.422485
0.80,0.20,0.923454,4.419485,0.981589,0.829917,0.894201
0.90,0.10,0.939229,5.348523,0.977988,0.843670,1.419460
0.95,0.05,0.948526,6.191603,0.982878,0.859349,1.996910
1.00,0.00,0.967533,17.398966,0.986285,0.883900,6.790151


In [20]:
results_scratio_det_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1).mean(0)

auc_score                     0.754594
non_abundant_over_abundant    5.183848
correct_sign_prop             0.887888
ks_D                          0.610485
w_distance                    1.468057
dtype: float64

In [21]:
results_scratio_det_per_run.groupby("metric").mean()

,value,run
metric,,
corr_auc_score_p,1.000000,1.0
corr_correct_sign_prop_p,0.952381,1.0
corr_ks_D_p,1.000000,1.0
corr_non_abundant_over_abundant_p,1.000000,1.0
corr_p_major_cluster_p,1.000000,1.0
corr_p_minor_cluster_p,-1.000000,1.0
corr_run_p,NaN,1.0
corr_w_distance_p,1.000000,1.0
mean_auc_score,0.951763,1.0


In [22]:
results_scratio_det_per_run.groupby("metric").std() / np.sqrt(3) 

,value,run
metric,,
corr_auc_score_p,0.000000,0.57735
corr_correct_sign_prop_p,0.013746,0.57735
corr_ks_D_p,0.000000,0.57735
corr_non_abundant_over_abundant_p,0.000000,0.57735
corr_p_major_cluster_p,0.000000,0.57735
corr_p_minor_cluster_p,0.000000,0.57735
corr_run_p,NaN,0.57735
corr_w_distance_p,0.000000,0.57735
mean_auc_score,0.001591,0.57735


## Deterministic sigma_min

In [23]:
scratio_det_sigma_min_result_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/scRatio/original_runs/deterministic_sigma_min")
results_scratio_det_sigma_min_per_oversamp, results_scratio_det_sigma_min_per_run = compute_evaluation_metrics(scratio_det_sigma_min_result_path, "scRatio", subsampling_rates, 3)

/tmp/ipykernel_793092/2231908646.py:132: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  results_per_run["value"].append(sp.stats.spearmanr(results_per_oversamp_df_i[metric],


In [24]:
results_scratio_det_sigma_min_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1)

,,auc_score,non_abundant_over_abundant,correct_sign_prop,ks_D,w_distance
p_major_cluster,p_minor_cluster,,,,,
0.50,0.50,0.304045,1.060681,0.461184,0.033369,0.003222
0.55,0.45,0.456587,1.569599,0.847225,0.250117,0.044847
0.60,0.40,0.827045,3.323208,0.957919,0.686245,0.204950
0.70,0.30,0.888541,3.679120,0.970416,0.772103,0.461216
0.80,0.20,0.932924,5.094938,0.974793,0.830584,0.902934
0.90,0.10,0.951199,6.287098,0.978571,0.861755,1.451174
0.95,0.05,0.957202,7.796914,0.979489,0.872898,2.063029
1.00,0.00,0.965416,20.099146,0.982031,0.882696,6.081147


In [25]:
results_scratio_det_sigma_min_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1).mean()

auc_score                     0.785370
non_abundant_over_abundant    6.113838
correct_sign_prop             0.893953
ks_D                          0.648721
w_distance                    1.401565
dtype: float64

In [26]:
results_scratio_det_sigma_min_per_run.groupby("metric").mean()

,value,run
metric,,
corr_auc_score_p,0.992063,1.0
corr_correct_sign_prop_p,0.952381,1.0
corr_ks_D_p,0.984127,1.0
corr_non_abundant_over_abundant_p,1.000000,1.0
corr_p_major_cluster_p,1.000000,1.0
corr_p_minor_cluster_p,-1.000000,1.0
corr_run_p,NaN,1.0
corr_w_distance_p,1.000000,1.0
mean_auc_score,0.957939,1.0


In [27]:
results_scratio_det_sigma_min_per_run.groupby("metric").std() / np.sqrt(3)

,value,run
metric,,
corr_auc_score_p,0.007937,0.57735
corr_correct_sign_prop_p,0.036370,0.57735
corr_ks_D_p,0.007937,0.57735
corr_non_abundant_over_abundant_p,0.000000,0.57735
corr_p_major_cluster_p,0.000000,0.57735
corr_p_minor_cluster_p,0.000000,0.57735
corr_run_p,NaN,0.57735
corr_w_distance_p,0.000000,0.57735
mean_auc_score,0.002098,0.57735


## Stochastic sigma_min

In [28]:
scratio_stochastic_result_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/scRatio/original_runs/stochastic")
results_scratio_stochastic_per_oversamp, results_scratio_stochastic_per_run = compute_evaluation_metrics(scratio_stochastic_result_path, "scRatio", subsampling_rates, 3)

/tmp/ipykernel_793092/2231908646.py:132: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  results_per_run["value"].append(sp.stats.spearmanr(results_per_oversamp_df_i[metric],


In [29]:
results_scratio_stochastic_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1)

,,auc_score,non_abundant_over_abundant,correct_sign_prop,ks_D,w_distance
p_major_cluster,p_minor_cluster,,,,,
0.50,0.50,0.318108,1.113850,0.450064,0.049455,0.010336
0.55,0.45,0.371005,1.292765,0.801649,0.146278,0.028062
0.60,0.40,0.684266,2.481976,0.934865,0.522809,0.179512
0.70,0.30,0.867581,3.466067,0.975711,0.750012,0.480440
0.80,0.20,0.915035,4.526102,0.977300,0.801831,0.885149
0.90,0.10,0.935480,5.112279,0.981942,0.837818,1.408231
0.95,0.05,0.948227,6.267302,0.984537,0.851117,2.158342
1.00,0.00,0.966776,16.811556,0.985667,0.881636,6.882879


In [30]:
results_scratio_stochastic_per_oversamp.groupby(["p_major_cluster", "p_minor_cluster"]).mean().drop("run", axis=1).mean(0)

auc_score                     0.750810
non_abundant_over_abundant    5.133987
correct_sign_prop             0.886467
ks_D                          0.605119
w_distance                    1.504119
dtype: float64

In [31]:
results_scratio_stochastic_per_run.groupby("metric").mean()

,value,run
metric,,
corr_auc_score_p,0.992063,1.0
corr_correct_sign_prop_p,0.976190,1.0
corr_ks_D_p,0.992063,1.0
corr_non_abundant_over_abundant_p,0.984127,1.0
corr_p_major_cluster_p,1.000000,1.0
corr_p_minor_cluster_p,-1.000000,1.0
corr_run_p,NaN,1.0
corr_w_distance_p,1.000000,1.0
mean_auc_score,0.950161,1.0


In [32]:
results_scratio_stochastic_per_run.groupby("metric").std() / np.sqrt(3)

,value,run
metric,,
corr_auc_score_p,0.007937,0.57735
corr_correct_sign_prop_p,0.000000,0.57735
corr_ks_D_p,0.007937,0.57735
corr_non_abundant_over_abundant_p,0.015873,0.57735
corr_p_major_cluster_p,0.000000,0.57735
corr_p_minor_cluster_p,0.000000,0.57735
corr_run_p,NaN,0.57735
corr_w_distance_p,0.000000,0.57735
mean_auc_score,0.001835,0.57735
